In [1]:
import sys
import os
from pathlib import Path
import random
import pandas as pd
import dash
from dash import dcc, html, Input, Output
import plotly.express as px
from pathlib import Path
from sklearn.decomposition import PCA

# Add source directory to path
sys.path.append(os.path.abspath("../src"))

# Import project utilities
from abstractionssymh.debug_utils import debug_info, debug_error, debug_success
from abstractionssymh.data_loader import parse_json_to_dsl
from abstractionssymh.plot_utils import plot_dsl_with_k3d, plot_dsl_with_matplotlib
from abstractionssymh.dsl_utils import collect_singleton_and_pair_data
from abstractionssymh.abstraction_utils import find_abstractions, Abstraction, integrate_abstractions

In [2]:
# --- Set Project Paths ---
current_path = Path.cwd()
base_project_dir = current_path.parent
dataset_directory = base_project_dir / "src" / "abstractionssymh" / "dataset"

saved_directory = base_project_dir / "src" / "abstractionssymh" / "saved"
saved_directory.mkdir(parents=True, exist_ok=True)

debug_info("Current notebook location:", current_path)
debug_info("Base project directory:", base_project_dir)
debug_info("Target dataset directory:", dataset_directory)
debug_info("Saved directory:", saved_directory)

[INFO] Current notebook location: C:\Users\Amogh\abstraction-discovery\notebooks
[INFO] Base project directory: C:\Users\Amogh\abstraction-discovery
[INFO] Target dataset directory: C:\Users\Amogh\abstraction-discovery\src\abstractionssymh\dataset
[INFO] Saved directory: C:\Users\Amogh\abstraction-discovery\src\abstractionssymh\saved


# LOAD DSL SHAPES

In [3]:
import pickle
from pathlib import Path

chair_directory = dataset_directory / "Chair"
pickle_file = saved_directory / "all_dsl_shapes.pkl"

# --- Check if pickle exists ---
if pickle_file.exists():
    print(f"Loading DSL shapes from pickle: {pickle_file}")
    with open(pickle_file, "rb") as f:
        all_dsl_shapes = pickle.load(f)
    print(f"Loaded {len(all_dsl_shapes)} DSL shapes from pickle.")
else:
    # --- Step 1: Load DSL objects from JSON files ---
    print("--- Step 1: Loading DSL objects from JSON files ---")
    json_files = sorted(chair_directory.glob("*.json"))
    all_dsl_shapes = {}

    for json_file in json_files:
        try:
            json_content = json_file.read_text(encoding="utf-8")
            dsl_obj = parse_json_to_dsl(json_content)
            all_dsl_shapes[json_file.name] = {
                "dsl": dsl_obj,
                "singleton_params": {},
                "pair_params": {},
            }
        except Exception as e:
            debug_error(f"Failed to load {json_file.name}: {e}")

    print(f"Loaded {len(all_dsl_shapes)} DSL shapes into memory.")

    # --- Step 2: Collect parameters ---
    print("\n--- Step 2: Collecting parameters for each shape ---")
    for name, data in all_dsl_shapes.items():
        dsl_obj = data["dsl"]
        singletons, pairs = collect_singleton_and_pair_data([dsl_obj])
        data["singleton_params"] = singletons
        data["pair_params"] = pairs

    debug_success("Populated singleton and pair parameters for each shape.")

    # --- Step 3: Save to pickle ---
    with open(pickle_file, "wb") as f:
        pickle.dump(all_dsl_shapes, f)
    print(f"Saved all_dsl_shapes with parameters to {pickle_file}")

Loading DSL shapes from pickle: C:\Users\Amogh\abstraction-discovery\src\abstractionssymh\saved\all_dsl_shapes.pkl
Loaded 6201 DSL shapes from pickle.


In [4]:
# all_dsl_shapes["Chair_1.json"].keys()

In [5]:
# all_dsl_shapes["Chair_1.json"]["dsl"]

In [6]:
# all_dsl_shapes["Chair_1.json"]["singleton_params"]

In [7]:
# all_dsl_shapes["Chair_1.json"]["pair_params"]

In [8]:
print(all_dsl_shapes["Chair_1.json"]["dsl"])

Union(
    Union(
        Translate(center=[0.003, 0.149, 0.072])
            Rotate(quat=[0.0000, 0.0000, 0.0000, 1.0000])
                Scale(lengths=[0.891, 0.224, 0.806])
                    Box(label=1),
        Translate(center=[0.004, -0.422, 0.170])
            Rotate(quat=[-0.3389, -0.0000, 0.0065, 0.9408])
                Scale(lengths=[0.898, 0.524, 1.154])
                    Box(label=2)
    ),
    Translate(center=[0.030, 0.488, -0.315])
        Rotate(quat=[-0.0821, -0.0071, -0.0494, 0.9954])
            Scale(lengths=[0.892, 0.781, 0.305])
                Box(label=0)
)


In [9]:
plot_dsl_with_k3d(all_dsl_shapes["Chair_220.json"]["dsl"])

[INFO] Expanding DSL tree for visualization...
[INFO] Found 9 total boxes after expansion.


Output()

[SUCCESS] 3D plot displayed successfully.


# EXTRACT FOR PLOTTING

In [10]:
import hdbscan
import pandas as pd
import copy

def cluster_key_with_hdbscan(data_dict, pattern_key, min_cluster_size=5, verbose=True):
    """
    Clusters the points for a given key (singleton or pair) using HDBSCAN.
    Noise points are stored separately. Logs cluster info if verbose=True.
    
    Args:
        data_dict (dict): Dictionary containing pattern records.
        pattern_key (str): Key in the dictionary to cluster.
        min_cluster_size (int): Minimum cluster size for HDBSCAN.
        cluster_params (list[str], optional): Subset of parameters to use for clustering. Defaults to all.
        verbose (bool): Whether to print clustering info.
        
    Returns:
        dict: Updated dictionary with original key removed, new cluster keys added, noise stored as key_noise.
    """
    data_dict = copy.deepcopy(data_dict)  # Avoid modifying the original dict
    
    if pattern_key not in data_dict:
        raise KeyError(f"'{pattern_key}' not found in dictionary.")
    
    records = data_dict.pop(pattern_key)  # Remove the original key
    df = pd.DataFrame(records)
    
    # Flatten parameters into columns
    param_df = pd.DataFrame(df['params'].to_list())
    
    # Run HDBSCAN clustering
    clusterer = hdbscan.HDBSCAN(min_cluster_size=min_cluster_size)
    cluster_labels = clusterer.fit_predict(param_df.values)
    
    unique_labels = sorted(set(cluster_labels))
    n_clusters = len([l for l in unique_labels if l != -1])
    n_noise = sum(cluster_labels == -1)
    
    if verbose:
        print(f"Clustering '{pattern_key}':")
        print(f"- Total points: {len(df)}")
        print(f"- Found clusters: {n_clusters}")
        print(f"- Noise points: {n_noise}")
    
    # Add points to new clusters and noise
    for cluster_id in unique_labels:
        cluster_records = df.iloc[cluster_labels == cluster_id].to_dict(orient='records')
        if cluster_id == -1:
            new_key = f"{pattern_key}_noise"
        else:
            new_key = f"{pattern_key}_cluster{cluster_id+1}"
        data_dict[new_key] = cluster_records
        if verbose:
            print(f"  -> {new_key}: {len(cluster_records)} points")
    
    return data_dict

In [11]:
# Paths for pickles
combined_singletons_detailed_pickle = saved_directory / "combined_singletons_detailed.pkl"
combined_pairs_detailed_pickle = saved_directory / "combined_pairs_detailed.pkl"

# --- Step 4: Load or build detailed dictionaries ---
if combined_singletons_detailed_pickle.exists() and combined_pairs_detailed_pickle.exists():
    print("--- Loading detailed dictionaries from pickle ---")
    with open(combined_singletons_detailed_pickle, "rb") as f:
        combined_singletons_detailed = pickle.load(f)
    with open(combined_pairs_detailed_pickle, "rb") as f:
        combined_pairs_detailed = pickle.load(f)
    print("Loaded detailed singleton and pair dictionaries from pickle.")
else:
    print("--- Building detailed dictionaries for singletons and pairs ---")
    combined_singletons_detailed = {}
    combined_pairs_detailed = {}

    for filename, data in all_dsl_shapes.items():
        # SINGLETON parameters
        for pattern_name, param_lists in data["singleton_params"].items():
            for param_list in param_lists or []:
                combined_singletons_detailed.setdefault(pattern_name, []).append({
                    'file': filename,
                    'params': param_list
                })

        # PAIR parameters
        for pattern_name, param_lists in data["pair_params"].items():
            for param_list in param_lists or []:
                combined_pairs_detailed.setdefault(pattern_name, []).append({
                    'file': filename,
                    'params': param_list
                })

    # Cluster Rotate
    combined_singletons_detailed = cluster_key_with_hdbscan(
        combined_singletons_detailed, 'Rotate', min_cluster_size=200, verbose=True
    )

    # Cluster Scale
    combined_singletons_detailed = cluster_key_with_hdbscan(
        combined_singletons_detailed, 'Scale', min_cluster_size=500, verbose=True
    )

    # Save to pickle for future runs
    with open(combined_singletons_detailed_pickle, "wb") as f:
        pickle.dump(combined_singletons_detailed, f)
    with open(combined_pairs_detailed_pickle, "wb") as f:
        pickle.dump(combined_pairs_detailed, f)

    print(f"Built {len(combined_singletons_detailed)} singleton patterns and {len(combined_pairs_detailed)} pair patterns.")
    debug_success("Detailed parameter structures for singletons and pairs created and saved successfully.")

--- Loading detailed dictionaries from pickle ---
Loaded detailed singleton and pair dictionaries from pickle.


In [12]:
combined_singletons_detailed.keys()

dict_keys(['Translate', 'SymRef', 'SymRot', 'SymTrans', 'Rotate_noise', 'Rotate_cluster1', 'Rotate_cluster2', 'Rotate_cluster3', 'Scale_noise', 'Scale_cluster1', 'Scale_cluster2', 'Scale_cluster3', 'Scale_cluster4', 'Scale_cluster5'])

In [13]:
# combined_singletons_detailed["Scale"][:5]

In [14]:
combined_pairs_detailed.keys()

dict_keys(['Rotate(Scale)', 'Scale(Box)', 'Translate(Rotate)', 'Union(Translate)', 'SymRef(Translate)', 'SymRef(Union)', 'Union(SymRef)', 'SymRot(Union)', 'Union(SymRot)', 'SymRot(Translate)', 'SymTrans(Translate)', 'Union(SymTrans)'])

In [15]:
# combined_pairs_detailed["Rotate(Scale)"][:5]

# SCATTER PLOT INTERACTIVE

In [16]:
import dash
from dash import dcc, html
from dash.dependencies import Input, Output
import pandas as pd
import plotly.express as px
from sklearn.decomposition import PCA
from abstractionssymh.plot_utils import plot_dsl_with_matplotlib_dash

# --- 1. PRECOMPUTE SINGLETON DATA (Simplified) ---
singleton_dfs = {}
for pattern_name, records in combined_singletons_detailed.items():
    df = pd.DataFrame(records)
    # We still need to unpack the params for the callback to use later
    param_cols = [f'param_{i}' for i in range(len(records[0]['params']))]
    params_df = pd.DataFrame(df['params'].to_list(), columns=param_cols)
    singleton_dfs[pattern_name] = pd.concat([df[['file']], params_df], axis=1)

# --- 2. PRECOMPUTE PAIR DATA (Simplified) ---
pair_dfs = {}
for pattern_name, records in combined_pairs_detailed.items():
    if not records:
        continue
    df = pd.DataFrame(records)
    param_cols = [f'param_{i}' for i in range(len(records[0]['params']))]
    params_df = pd.DataFrame(df['params'].to_list(), columns=param_cols)
    pair_dfs[pattern_name] = pd.concat([df[['file']], params_df], axis=1)

# --- 3. AVAILABLE PATTERNS ---
pattern_types = ['Singleton', 'Pair']

# --- 4. INITIALIZE DASH APP ---
app = dash.Dash(__name__)

# --- 5. LAYOUT ---
app.layout = html.Div(
    style={'backgroundColor': '#1e1e1e', 'color': '#fff', 'padding': '30px', 'fontFamily': 'Arial, sans-serif', 'minHeight': '100vh'},
    children=[
        html.H1("Interactive DSL Pattern Explorer",
                style={'textAlign': 'center', 'marginBottom': '40px', 'color': '#f0a500', 'fontSize': '36px'}),

        # Pattern type dropdown
        html.Div(
            style={'display': 'flex', 'alignItems': 'center', 'justifyContent': 'center', 'marginBottom': '20px', 'gap': '15px'},
            children=[
                html.Label("Pattern Type:", style={'fontSize': '18px', 'fontWeight': 'bold'}),
                dcc.Dropdown(
                    id='pattern-type-dropdown',
                    options=[{'label': t, 'value': t} for t in pattern_types],
                    value='Singleton',
                    style={'color': '#1e1e1e', 'width': '200px', 'borderRadius': '5px'}
                ),
                html.Label("Select Pattern:", style={'fontSize': '18px', 'fontWeight': 'bold'}),
                dcc.Dropdown(id='pattern-selector-dropdown', style={'color': '#1e1e1e', 'width': '300px', 'borderRadius': '5px'})
            ]
        ),

        # Slider
        html.Div(
            style={'marginBottom': '30px', 'textAlign': 'center'},
            children=[
                html.Label("Percentage of points to display:", style={'fontSize': '18px', 'fontWeight': 'bold'}),
                dcc.Slider(
                    id='percent-slider',
                    min=10, max=100, step=10, value=100,
                    marks={i: f'{i}%' for i in range(10, 110, 10)},
                    tooltip={"placement": "bottom", "always_visible": True},
                ),
                html.Div(id='point-count-display', style={'marginTop': '10px', 'fontSize': '16px', 'color': '#f0a500'})
            ]
        ),

        # Main content
        html.Div(
            style={'display': 'flex', 'gap': '20px', 'flexWrap': 'wrap', 'justifyContent': 'center'},
            children=[
                html.Div(
                    style={'flex': '1 1 500px', 'maxWidth': '800px', 'minWidth': '300px',
                           'backgroundColor': '#2b2b2b', 'padding': '15px', 'borderRadius': '10px',
                           'boxShadow': '0 4px 10px rgba(0,0,0,0.3)'},
                    children=[dcc.Graph(id='scatter-plot', style={'height': '600px', 'borderRadius': '10px'})]
                ),
                html.Div(
                    style={'flex': '1 1 500px', 'maxWidth': '800px', 'minWidth': '300px',
                           'backgroundColor': '#2b2b2b', 'padding': '15px', 'borderRadius': '10px',
                           'boxShadow': '0 4px 10px rgba(0,0,0,0.3)',
                           'display': 'flex', 'flexDirection': 'column', 'alignItems': 'center'},
                    children=[
                        html.H3("Selected Chair Visualization", style={'marginTop': '0px', 'color': '#f0a500'}),
                        html.Img(id='dsl-plot-image', style={'width': '100%', 'height': '600px', 'objectFit': 'contain'})
                    ]
                )
            ]
        )
    ]
)

# --- 6. CALLBACKS ---

# Update pattern selector based on type
@app.callback(
    Output('pattern-selector-dropdown', 'options'),
    Output('pattern-selector-dropdown', 'value'),
    Input('pattern-type-dropdown', 'value')
)
def update_pattern_options(pattern_type):
    if pattern_type == 'Singleton':
        options = [{'label': name, 'value': name} for name in singleton_dfs.keys()]
    else:
        options = [{'label': name, 'value': name} for name in pair_dfs.keys()]
    value = options[0]['value'] if options else None
    return options, value

# Update scatter plot
@app.callback(
    [Output('scatter-plot', 'figure'),
     Output('point-count-display', 'children')],
    [Input('pattern-type-dropdown', 'value'),
     Input('pattern-selector-dropdown', 'value'),
     Input('percent-slider', 'value')]
)
def update_scatter(pattern_type, selected_pattern, percent):
    if not selected_pattern:
        return {}, "Please select a pattern"

    # --- Step 1: Get the correct raw DataFrame ---
    if pattern_type == 'Singleton':
        if selected_pattern not in singleton_dfs: return {}, ""
        df = singleton_dfs[selected_pattern]
    else: # Pair
        if selected_pattern not in pair_dfs: return {}, ""
        df = pair_dfs[selected_pattern]

    # --- Step 2: Perform PCA and projection ON-THE-FLY ---
    param_cols = [c for c in df.columns if c.startswith('param_')]
    n_dims = len(param_cols)
    
    if n_dims > 3:
        pca = PCA(n_components=3)
        reduced_data = pca.fit_transform(df[param_cols].values)
        df_plot = pd.DataFrame(reduced_data, columns=['PC1', 'PC2', 'PC3'])
    else:
        # If 3 or fewer dimensions, just rename columns
        df_plot = df.rename(columns={f'param_{i}': f'PC{i+1}' for i in range(n_dims)})
        for pc in ['PC1','PC2','PC3']:
            if pc not in df_plot: df_plot[pc] = 0
            
    df_plot['file'] = df['file'].values

    # --- Step 3: Sample the data ---
    n_total = len(df_plot)
    n_sample = max(1, int(n_total * percent / 100))
    df_sampled = df_plot.sample(n=n_sample, random_state=42)
    
    # --- Step 4: Create the figure ---
    title = f'"{selected_pattern}" Parameters ({n_dims}D{" -> 3D PCA Projection" if n_dims > 3 else ""})'
    fig = px.scatter_3d(df_sampled, x='PC1', y='PC2', z='PC3', hover_data=['file'], custom_data=['file'])
    fig.update_layout(template='plotly_dark', title=title)
    fig.update_traces(marker=dict(size=4, opacity=0.7))

    point_text = f"Showing {n_sample:,} of {n_total:,} points ({percent}%)"
    return fig, point_text

# Update image
@app.callback(
    Output('dsl-plot-image', 'src'),
    Input('scatter-plot', 'clickData')
)
def update_image(clickData):
    if clickData is None:
        return dash.no_update
    clicked_filename = clickData['points'][0]['customdata'][0]
    dsl_obj = all_dsl_shapes[clicked_filename]["dsl"]
    return plot_dsl_with_matplotlib_dash(dsl_obj)

# --- 7. RUN APP ---
app.run(debug=True, port=8052)

In [ ]:
import torch
from pathlib import Path
import re

from abstractionssymh.abstraction_utils import DEVICE, Autoencoder, find_abstractions
from abstractionssymh.debug_utils import debug_info, debug_success, debug_error

# --- Helper to make filenames safe ---
def make_safe_filename(name: str) -> str:
    # Replace any non-word or non-hyphen character with underscore
    safe = re.sub(r'[^\w\-]+', '_', name)
    # Collapse multiple underscores
    safe = re.sub(r'_+', '_', safe)
    # Strip leading/trailing underscores
    safe = safe.strip('_')
    # Lowercase and add extension
    return safe.lower() + ".pth"

# --- A. Prepare Data for Training (Preserve file info) ---
debug_info("--- Preparing data for model training ---")

training_singleton_params = {}
training_singleton_files = {}
for pattern_name, records in combined_singletons_detailed.items():
    training_singleton_params[pattern_name] = [rec['params'] for rec in records]
    training_singleton_files[pattern_name] = [rec['file'] for rec in records]

training_pair_params = {}
training_pair_files = {}
for pattern_name, records in combined_pairs_detailed.items():
    training_pair_params[pattern_name] = [rec['params'] for rec in records]
    training_pair_files[pattern_name] = [rec['file'] for rec in records]

debug_success("Data flattened for autoencoder training, file references preserved.")

In [ ]:
training_singleton_params.keys()

In [ ]:
# --- B. Train or Load Models ---
models_directory = saved_directory / "models"
models_directory.mkdir(parents=True, exist_ok=True)
models_exist = any(models_directory.iterdir())

singleton_models = {}
pair_models = {}

if models_exist:
    debug_info("--- Models found on disk. Loading from files... ---")

    # --- LOAD SINGLETON MODELS ---
    for name, params in training_singleton_params.items():
        if len(params) < 50: 
            continue
        num_params = len(params[0])
        if num_params <= 1: 
            continue

        save_file = models_directory / make_safe_filename(name)
        if save_file.is_file():
            try:
                model = Autoencoder(num_params, max(1, num_params - 1)).to(DEVICE)
                model.load_state_dict(torch.load(save_file, map_location=DEVICE))
                model.eval()
                singleton_models[name] = model
                debug_success(f"Loaded singleton model for '{name}'")
            except Exception as e:
                debug_error(f"Failed to load singleton model for '{name}': {e}")

    # --- LOAD PAIR MODELS ---
    for name, params in training_pair_params.items():
        if len(params) < 50: 
            continue
        num_params = len(params[0])
        if num_params <= 1: 
            continue

        save_file = models_directory / make_safe_filename(name)
        if save_file.is_file():
            try:
                model = Autoencoder(num_params, max(1, num_params - 1)).to(DEVICE)
                model.load_state_dict(torch.load(save_file, map_location=DEVICE))
                model.eval()
                pair_models[name] = model
                debug_success(f"Loaded pair model for '{name}'")
            except Exception as e:
                debug_error(f"Failed to load pair model for '{name}': {e}")

else:
    debug_info("--- No models found. Starting training process... ---")

    debug_info("Training singleton models...")
    singleton_models = find_abstractions(
        training_singleton_params, 
        structure_type="SINGLETONS", 
        min_examples=50, 
        epochs=20
    )

    debug_info("Training pair models...")
    pair_models = find_abstractions(
        training_pair_params, 
        structure_type="PAIRS", 
        min_examples=50, 
        epochs=20
    )

    # --- SAVE MODELS ---
    for name, model in singleton_models.items():
        torch.save(model.state_dict(), models_directory / make_safe_filename(name))
        debug_success(f"Saved singleton model for '{name}'")

    for name, model in pair_models.items():
        torch.save(model.state_dict(), models_directory / make_safe_filename(name))
        debug_success(f"Saved pair model for '{name}'")

debug_success(
    f"--- Workflow complete. {len(singleton_models)} singleton and {len(pair_models)} pair models are ready. ---"
)

In [ ]:
singleton_models.keys()

In [ ]:
pair_models.keys()

In [ ]:
# --- 4. INITIALIZE DASH APP ---
app = dash.Dash(__name__)

# --- 5. LAYOUT ---
app.layout = html.Div(
    style={'backgroundColor': '#1e1e1e', 'color': '#fff', 'padding': '30px', 'fontFamily': 'Arial, sans-serif'},
    children=[
        html.H1("Interactive DSL Abstraction Explorer",
                style={'textAlign': 'center', 'marginBottom': '40px', 'color': '#f0a500'}),

        # --- Top Controls: Dropdowns and Slider ---
        html.Div(
            style={'display': 'flex', 'alignItems': 'center', 'justifyContent': 'center', 'marginBottom': '20px', 'gap': '15px'},
            children=[
                html.Label("Pattern Type:"),
                dcc.Dropdown(
                    id='pattern-type-dropdown',
                    options=[{'label': t, 'value': t} for t in pattern_types],
                    value='Singleton',
                    style={'color': '#1e1e1e', 'width': '200px'}
                ),
                html.Label("Select Pattern:"),
                dcc.Dropdown(id='pattern-selector-dropdown', style={'color': '#1e1e1e', 'width': '300px'})
            ]
        ),
        html.Div(
            style={'maxWidth': '800px', 'margin': '0 auto 30px auto'},
            children=[
                html.Label("Percentage of points to display:"),
                dcc.Slider(
                    id='percent-slider',
                    min=10, max=100, step=10, value=100,
                    marks={i: f'{i}%' for i in range(10, 110, 10)},
                    tooltip={"placement": "bottom", "always_visible": True},
                ),
                html.Div(id='point-count-display', style={'textAlign': 'center', 'marginTop': '10px', 'color': '#f0a500'})
            ]
        ),

        # --- Main Scatter Plot ---
        html.Div(
            style={
                'backgroundColor': '#2b2b2b', 'padding': '15px', 'borderRadius': '10px',
                'boxShadow': '0 4px 10px rgba(0,0,0,0.3)', 'marginBottom': '30px'
            },
            children=[dcc.Graph(id='scatter-plot', style={'height': '600px'})]
        ),

        # --- Bottom Two-Column Layout for DSL Details ---
        html.Div(
            style={'display': 'flex', 'gap': '30px', 'justifyContent': 'center'},
            children=[
                # Left Column: Original DSL
                html.Div(
                    style={
                        'flex': '1', 'maxWidth': '700px', 'backgroundColor': '#2b2b2b',
                        'padding': '20px', 'borderRadius': '10px', 'boxShadow': '0 4px 10px rgba(0,0,0,0.3)'
                    },
                    children=[
                        html.H4("Selected Chair (Original DSL)", style={"color": "#f0a500", "textAlign": "center"}),
                        html.Pre(id='selected-dsl-text', style={'whiteSpace': 'pre-wrap', 'wordWrap': 'break-word', 'backgroundColor': '#1e1e1e', 'padding': '10px', 'borderRadius': '5px', 'minHeight': '150px'}),
                        html.Img(id='selected-dsl-plot', style={'width': '100%', 'height': '400px', 'objectFit': 'contain', 'marginTop': '15px'})
                    ]
                ),
                # Right Column: Abstracted DSL
                html.Div(
                    style={
                        'flex': '1', 'maxWidth': '700px', 'backgroundColor': '#2b2b2b',
                        'padding': '20px', 'borderRadius': '10px', 'boxShadow': '0 4px 10px rgba(0,0,0,0.3)'
                    },
                    children=[
                        html.H4("Abstracted Version", style={"color": "#f0a500", "textAlign": "center"}),
                        html.Pre(id='abstracted-dsl-text', style={'whiteSpace': 'pre-wrap', 'wordWrap': 'break-word', 'backgroundColor': '#1e1e1e', 'padding': '10px', 'borderRadius': '5px', 'minHeight': '150px'}),
                        html.Img(id='abstracted-dsl-plot', style={'width': '100%', 'height': '400px', 'objectFit': 'contain', 'marginTop': '15px'})
                    ]
                )
            ]
        )
    ]
)

# --- 6. CALLBACKS ---

# Callback 1: Update pattern selector dropdown based on pattern type
@app.callback(
    Output('pattern-selector-dropdown', 'options'),
    Output('pattern-selector-dropdown', 'value'),
    Input('pattern-type-dropdown', 'value')
)
def update_pattern_options(pattern_type):
    if pattern_type == 'Singleton':
        options = [{'label': name, 'value': name} for name in singleton_dfs.keys()]
    else:
        options = [{'label': name, 'value': name} for name in pair_dfs.keys()]
    value = options[0]['value'] if options else None
    return options, value

# Callback 2: Update the main scatter plot
@app.callback(
    Output('scatter-plot', 'figure'),
    Output('point-count-display', 'children'),
    Input('pattern-type-dropdown', 'value'),
    Input('pattern-selector-dropdown', 'value'),
    Input('percent-slider', 'value')
)
def update_scatter(pattern_type, selected_pattern, percent):
    if not selected_pattern:
        return {}, "Please select a pattern"

    df = None
    if pattern_type == 'Singleton':
        if selected_pattern in singleton_dfs:
            df = singleton_dfs[selected_pattern]
    else: # Pair type
        # MODIFIED: Get data from pair_dfs instead of projection_cache
        if selected_pattern in pair_dfs:
            df = pair_dfs[selected_pattern]

    if df is None:
        raise dash.exceptions.PreventUpdate

    # --- This logic is now the same for both Singletons and Pairs ---
    param_cols = [c for c in df.columns if c.startswith('param_')]
    n_dims = len(param_cols)
    n_total = len(df)
    n_sample = max(1, int(n_total * percent / 100))
    # Note: Sampling before PCA for performance on large datasets
    df_sampled = df.sample(n=n_sample, random_state=42)

    if n_dims > 3:
        pca = PCA(n_components=3)
        reduced_data = pca.fit_transform(df_sampled[param_cols].values)
        df_plot = pd.DataFrame(reduced_data, columns=['PC1', 'PC2', 'PC3'])
    else:
        df_plot = df_sampled.rename(columns={f'param_{i}': f'PC{i+1}' for i in range(n_dims)})
        for pc in ['PC1', 'PC2', 'PC3']:
            if pc not in df_plot: df_plot[pc] = 0

    df_plot['file'] = df_sampled['file'].values

    title = f'"{selected_pattern}" Parameters ({n_dims}D{" -> 3D PCA Projection" if n_dims > 3 else ""})'
    fig = px.scatter_3d(df_plot, x='PC1', y='PC2', z='PC3', hover_data=['file'], custom_data=['file'])
    fig.update_layout(template='plotly_dark', title={'text': title, 'x': 0.5})
    fig.update_traces(marker=dict(size=4, opacity=0.8))

    point_text = f"Showing {n_sample:,} of {n_total:,} points ({percent}%)"
    return fig, point_text

# Callback 3: Update the bottom two columns when a point is clicked on the scatter plot
@app.callback(
    Output('selected-dsl-text', 'children'),
    Output('selected-dsl-plot', 'src'),
    Output('abstracted-dsl-text', 'children'),
    Output('abstracted-dsl-plot', 'src'),
    Input('scatter-plot', 'clickData')
)
def update_details_on_click(clickData):
    if clickData is None:
        # Prevents the callback from firing on initial load
        return "Click a point on the scatter plot to see details.", "", "Details will appear here.", ""

    # --- Get Original DSL ---
    clicked_filename = clickData['points'][0]['customdata'][0]
    print(clicked_filename)
    original_dsl = all_dsl_shapes[clicked_filename]["dsl"]

    # --- Get Abstracted DSL ---
    abstracted_dsl = integrate_abstractions(
        original_dsl, singleton_models, pair_models, error_threshold=0.01
    )

    # --- Generate Plots ---
    orig_src = plot_dsl_with_matplotlib_dash(original_dsl)
    abs_src = plot_dsl_with_matplotlib_dash(abstracted_dsl)

    return str(original_dsl), orig_src, str(abstracted_dsl), abs_src

# --- 7. RUN APP ---
if __name__ == '__main__':
    app.run(debug=True)

# EDITABLE ABSTRACTIONS

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import numpy as np

from abstractionssymh.dsl_nodes import (
    Box,
    Scale,
    Rotate,
    Translate,
    Union,
    SymRef,
    SymRot,
    SymTrans,
)

# ==============================================================================
# --- PART 1: DSL <-> DICTIONARY CONVERSION (Your working versions) ---
# ==============================================================================

def dsl_to_dict(node):
    """Converts your DSL node tree into a nested dictionary."""
    node_type = type(node).__name__
    if isinstance(node, Box):
        params = {"label": node.label}
        children = []
    elif isinstance(node, Scale):
        params = {"lengths": node.lengths.tolist()}
        children = [dsl_to_dict(node.child)]
    elif isinstance(node, Rotate):
        params = {"quaternion": node.quaternion.tolist()}
        children = [dsl_to_dict(node.child)]
    elif isinstance(node, Translate):
        params = {"center": node.center.tolist()}
        children = [dsl_to_dict(node.child)]
    elif isinstance(node, Union):
        params = {}
        children = [dsl_to_dict(node.left), dsl_to_dict(node.right)]
    elif isinstance(node, SymRef):
        params = {"plane": node.plane.tolist(), "point_on_plane": node.point_on_plane.tolist()}
        children = [dsl_to_dict(node.child)]
    elif isinstance(node, SymRot):
        params = {"axis": node.axis.tolist(), "center": node.center.tolist(), "n": node.n}
        children = [dsl_to_dict(node.child)]
    elif isinstance(node, SymTrans):
        params = {"end_point": node.end_point.tolist(), "n": node.n}
        children = [dsl_to_dict(node.child)]
    elif isinstance(node, Abstraction):
        params = {"pattern_name": node.pattern_name, "compressed_params": node.compressed_params}
        children = [dsl_to_dict(c) for c in node.children]
    else:
        params = {}
        children = []
    return {"type": node_type, "params": params, "children": children}

def dict_to_dsl(d, singleton_models=None, pair_models=None):
    """Converts a dictionary back into a DSL object using your trained models."""
    singleton_models = singleton_models or {}
    pair_models = pair_models or {}
    node_type = d["type"]
    params = d.get("params", {})
    children = [dict_to_dsl(c, singleton_models, pair_models) for c in d.get("children", [])]

    if node_type == "Abstraction":
        pattern_name = params.get("pattern_name")
        model = singleton_models.get(pattern_name) or pair_models.get(pattern_name)
        if model is None:
            raise ValueError(f"No trained model found for Abstraction '{pattern_name}'")
        return Abstraction(pattern_name, params["compressed_params"], model=model, children=children)
    elif node_type == "Box":
        return Box(params["label"])
    elif node_type == "Scale":
        return Scale(children[0], params["lengths"])
    elif node_type == "Rotate":
        return Rotate(children[0], params["quaternion"])
    elif node_type == "Translate":
        return Translate(children[0], params["center"])
    elif node_type == "SymRef":
        return SymRef(children[0], plane_normal=params["plane"], point_on_plane=params["point_on_plane"])
    elif node_type == "SymRot":
        return SymRot(children[0], axis=params["axis"], center=params["center"], n_fold=params["n"])
    elif node_type == "SymTrans":
        return SymTrans(children[0], end_point=params["end_point"], n_fold=params["n"])
    elif node_type == "Union":
        return Union(children[0], children[1])
    else:
        raise ValueError(f"Unknown node type in dict: {node_type}")

In [ ]:
# selected_chair_dsl = all_dsl_shapes["Chair_1.json"]["dsl"] # The first chair
# selected_chair_dsl = all_dsl_shapes["Chair_274.json"]["dsl"] # Good Example
# selected_chair_dsl = all_dsl_shapes["Chair_300.json"]["dsl"] # Bad Example
selected_chair_dsl = all_dsl_shapes["Chair_2796.json"]["dsl"] # Good Example
# selected_chair_dsl = all_dsl_shapes["Chair_3210.json"]["dsl"] # Good Example

# selected_chair_dsl = all_dsl_shapes["Chair_1038.json"]["dsl"]
# selected_chair_dsl = all_dsl_shapes["Chair_3096.json"]["dsl"]

# --- 2. Run the Abstraction Process ---
# This function walks the tree and replaces patterns with 'Abstraction' nodes
abstracted_chair_dsl = integrate_abstractions(
    selected_chair_dsl, 
    singleton_models, 
    pair_models, 
    error_threshold=0.01  # A stricter threshold for high-quality abstractions
)

# --- 3. Display Textual "Before and After" Comparison ---
# print("\n--- ORIGINAL CHAIR DSL ---")
# print(selected_chair_dsl)

# print("\n--- ABSTRACTED CHAIR DSL ---")
# print(abstracted_chair_dsl)

# ==============================================================================
# --- PART 2: THE NEW, INTUITIVE ACCORDION-ONLY WIDGET EDITOR ---
# ==============================================================================

live_output = widgets.Output()

def build_accordion_editor(data_source, update_fn):
    """
    (NEW) Recursively builds a clean, intuitive editor using only
    collapsible accordions and simple text boxes for parameters.
    """
    # --- 1. Parameter Widgets (using classic HBox with FloatText) ---
    param_widgets = []
    params_dict = data_source.get('params', {})
    for key, value in params_dict.items():
        hbox_items = [widgets.Label(value=f"{key}:", layout=widgets.Layout(width='140px'))]
        if isinstance(value, (int, float)):
            widget = widgets.FloatText(value=value, layout=widgets.Layout(width='100px'), step=0.01)
            def make_observer(source_dict, key):
                def on_value_change(change):
                    source_dict[key] = change.new
                    update_fn()
                return on_value_change
            widget.observe(make_observer(params_dict, key), names='value')
            hbox_items.append(widget)
        elif isinstance(value, list) and all(isinstance(v, (int, float)) for v in value):
            for i, num in enumerate(value):
                widget = widgets.FloatText(value=num, layout=widgets.Layout(width='100px'), step=0.01)
                def make_observer(source_list, index):
                    def on_value_change(change):
                        source_list[index] = change.new
                        update_fn()
                    return on_value_change
                widget.observe(make_observer(value, i), names='value')
                hbox_items.append(widget)
        else:
            hbox_items.append(widgets.Label(str(value)))
        param_widgets.append(widgets.HBox(hbox_items))
    
    param_box = widgets.VBox(param_widgets, layout=widgets.Layout(padding='5px 0 10px 0'))

    # --- 2. Children Node Widgets ---
    children_widgets = []
    children_list = data_source.get('children', [])
    for child_data in children_list:
        children_widgets.append(build_accordion_editor(child_data, update_fn))
    
    # --- 3. Assemble the Node Widget ---
    # Put parameters first, then the list of children widgets.
    node_content = widgets.VBox([param_box] + children_widgets)
    
    # Wrap the whole thing in a final Accordion for collapsibility.
    node_accordion = widgets.Accordion(children=[node_content])
    node_accordion.set_title(0, f"{data_source.get('type', 'Node')}")
    # Start with all nodes collapsed for a clean initial view.
    node_accordion.selected_index = None 
    
    return node_accordion

# ==============================================================================
# --- USAGE ---
# ==============================================================================

# Assume 'abstracted_chair_dsl' is your DSL object.
# Assume 'singleton_models' and 'pair_models' are your trained model dictionaries.

# 1. Convert DSL to dict
dsl_dict = dsl_to_dict(abstracted_chair_dsl)

# 2. Define the update function
def trigger_update():
    with live_output:
        clear_output(wait=True)
        # Rebuild DSL using your actual trained models
        new_dsl = dict_to_dsl(dsl_dict, singleton_models=singleton_models, pair_models=pair_models)
        plot_dsl_with_k3d(new_dsl) # Your plotting function

# 3. Build the new accordion-only editor
editor_widget = build_accordion_editor(dsl_dict, trigger_update)
editor_widget.layout = widgets.Layout(width='50%', border='1px solid #ddd', padding='5px')
live_output.layout = widgets.Layout(width='50%', padding='10px')

# 4. Display in a side-by-side layout
print("Interactive DSL Editor and Live View")
display(widgets.HBox([editor_widget, live_output]))
trigger_update()


In [ ]:
# bruh

In [ ]:
# import pandas as pd
# from tqdm.notebook import tqdm

# from abstractionssymh.abstraction_compare_utils import calculate_tree_complexity_reduction, calculate_parameter_compression, calculate_abstraction_coverage
# from abstractionssymh.abstraction_compare_utils import get_point_cloud_from_dsl, calculate_chamfer_distance, plot_point_clouds_with_k3d

# all_results_with_objects = []

# # Loop through all shapes with a progress bar
# for chair_id, original_chair in tqdm(list(all_dsl_shapes.items())[:20], desc="Processing and Storing Chairs"):
#     # 1. Perform the abstraction
#     abstracted_chair = integrate_abstractions(
#         original_chair["dsl"], singleton_models, pair_models, error_threshold=0.01
#     )
    
#     # 2. Calculate metrics
#     complexity = calculate_tree_complexity_reduction(original_chair["dsl"], abstracted_chair)
#     compression = calculate_parameter_compression(original_chair["dsl"], abstracted_chair)
#     coverage = calculate_abstraction_coverage(original_chair, abstracted_chair)

#     pc_original = get_point_cloud_from_dsl(original_chair["dsl"], points_per_box=1000)
#     pc_abstracted = get_point_cloud_from_dsl(abstracted_chair, points_per_box=1000)
#     chamfer_distance = calculate_chamfer_distance(pc_original, pc_abstracted)
    
#     # 3. Store metrics AND the actual DSL objects
#     chair_result = {
#         'chair_id': chair_id,
#         # --- Metrics ---
#         'node_reduction_%': float(complexity['metrics']['node_count_reduction'].strip('%')),
#         'depth_reduction_%': float(complexity['metrics']['max_depth_reduction'].strip('%')),
#         'param_compression_%': float(compression['metrics']['parameter_compression_ratio'].strip('%')),
#         'abstraction_coverage_%': float(coverage['metrics']['abstraction_coverage'].strip('%')),
#         'chamfer_distance': float(chamfer_distance),
#         # --- ADDING THE DSL OBJECTS ---
#         'original_chair_obj': original_chair,
#         'abstracted_chair_obj': abstracted_chair
#     }
    
#     all_results_with_objects.append(chair_result)

# print(f"\nProcessed and stored {len(all_results_with_objects)} chairs with their objects and metrics.")

# # --- Create the DataFrame ---
# # The DataFrame will now have columns containing the chair objects themselves.
# results_df_with_objects = pd.DataFrame(all_results_with_objects)

# # Display the DataFrame columns to confirm objects are stored
# print("\nDataFrame columns:", results_df_with_objects.columns.tolist())
# display(results_df_with_objects.head())

In [ ]:
# results_df_with_objects.to_csv(saved_directory / Path("abstraction_results.csv"))
# results_df_with_objects.to_pickle(saved_directory / Path("abstraction_results.pkl"))

In [ ]:
import pandas as pd

# --- LOADING THE RESULTS ---
# Load the entire DataFrame, including the DSL objects, back from the pickle file
loaded_results_df = pd.read_pickle(saved_directory / Path("abstraction_results.pkl"))

print("Successfully loaded the results DataFrame.")
print("Columns:", loaded_results_df.columns.tolist())
display(loaded_results_df.head())

# Now you can access the objects just like before
a_random_chair = loaded_results_df.sample().iloc[0]
original_obj = a_random_chair['original_chair_obj']

print("\nSuccessfully accessed a loaded DSL object:")
print(original_obj)

In [ ]:
combined_singletons_detailed.keys()

In [ ]:
import torch
import pandas as pd
import itertools

# ==============================================================================
# 1. LANGUAGE DEFINITIONS: Operations and Constants
# ==============================================================================

class FloatOperation:
    """A simple class to wrap mathematical operations."""
    def __init__(self, func, unary: bool = False):
        self.func = func
        self.unary = unary

LANGUAGE = {
    "binary_ops": {
        "Add": FloatOperation(lambda a, b: a + b),
        "Sub": FloatOperation(lambda a, b: a - b),
        "Mul": FloatOperation(lambda a, b: a * b),
        "Div": FloatOperation(lambda a, b: a / (b + 1e-6)),  # avoid divide by zero
    },
    "constants": [0.5, 1.0, 2.0, -1.0, 3.1415],
}

UNARY_OPERATIONS = {
    "Neg": FloatOperation(lambda a: -a, unary=True),
    "Inv": FloatOperation(lambda a: 1.0 / (a + 1e-6), unary=True)
}

# ==============================================================================
# 2. DATA PREPARATION (Dummy Function)
# ==============================================================================

def prepare_pattern_data(pattern_name: str, raw_records: list) -> pd.DataFrame:
    """Prepares and validates pattern data from raw records into a DataFrame."""
    PATTERN_DIMENSIONS = {"Scale": 3, "Translate": 3, "Rotate": 4, "SymRef": 6}
    expected_dim = PATTERN_DIMENSIONS.get(pattern_name)
    if expected_dim is None:
        raise ValueError(f"Pattern '{pattern_name}' is not defined.")

    valid_params = [r["params"] for r in raw_records if len(r["params"]) == expected_dim]
    if not valid_params:
        raise ValueError(f"No valid records found for '{pattern_name}'.")

    param_names = [f"param_{i}" for i in range(expected_dim)]
    return pd.DataFrame(valid_params, columns=param_names)

# ==============================================================================
# 3. MATCHING UTILITIES
# ==============================================================================

def compare_with_tolerance(predicted: torch.Tensor, target: torch.Tensor,
                           max_error: float, min_fraction: float):
    """Compares a single predicted tensor with the target tensor."""
    if predicted.shape != target.shape:
        # Handle scalar expansion
        if predicted.ndim == 0:
            predicted = predicted.unsqueeze(0).expand_as(target)
        else:
            return False, []

    matches = torch.abs(predicted - target) < max_error
    fraction = matches.sum().item() / len(target)

    if fraction >= min_fraction:
        return True, torch.where(matches)[0].tolist()
    return False, []

def batch_compare(pred_batches: torch.Tensor, target: torch.Tensor,
                  max_error: float, min_fraction: float,
                  index_pairs, variable_names):
    """Compares a batch of predicted tensors with the target tensor."""
    results = []
    for i in range(pred_batches.shape[0]):
        success, matched_idx = compare_with_tolerance(pred_batches[i], target, max_error, min_fraction)
        if success:
            # Map indices back to variable names
            expr_vars = tuple(variable_names[j] for j in index_pairs[i])
            results.append((expr_vars, matched_idx))
    return results

# ==============================================================================
# 4. SEARCH FUNCTIONS
# ==============================================================================

def search_direct_variable(inp, return_indices=False):
    """Searches if the target can be explained by a single input variable."""
    matches = []
    for name, values in inp["variables"].items():
        success, idx = compare_with_tolerance(values, inp["target"], inp["tol"], inp["min_match"])
        if success:
            matches.append((name, idx) if return_indices else name)
    return matches

def search_constants(inp, return_indices=False):
    """Searches if the target can be explained by a single constant value."""
    matches = []
    tolerance = 0.01
    for name, values in inp["simple_consts"].items():
        success, idx = compare_with_tolerance(values, inp["target"], tolerance, inp["min_match"])
        if success:
            matches.append((name, idx) if return_indices else name)
    return matches

def search_unary_operations(inp, return_indices=False):
    """Searches for expressions using one unary operator on a single variable."""
    matches = []
    for op_name, op in inp["unary_ops"].items():
        for var_name, var_value in inp["variables"].items():
            prediction = op.func(var_value)
            success, idx = compare_with_tolerance(prediction, inp["target"], inp["tol"], inp["min_match"])
            if success:
                expr = f"{op_name}({var_name})"
                matches.append((expr, idx) if return_indices else expr)
    return matches

def search_binary_operations(inp, return_indices=False):
    """Searches for expressions with one binary operator."""
    matches = []
    var_keys = list(inp["variables"].keys())
    const_keys = list(inp["extended_consts"].keys())
    all_keys = var_keys + const_keys

    var_values = list(inp["variables"].values())
    if not var_values: return []

    const_values = [c.expand(len(inp["target"])) for c in inp["extended_consts"].values()]
    value_stack = torch.stack(var_values + const_values)

    index_pairs = torch.cartesian_prod(torch.arange(len(var_values)), torch.arange(len(all_keys)))
    candidate_pairs = value_stack[index_pairs]

    for op_name, op in inp["binary_ops"].items():
        results = op.func(candidate_pairs[:, 0, :], candidate_pairs[:, 1, :])
        comparisons = batch_compare(results, inp["target"], inp["tol"], inp["min_match"], index_pairs, all_keys)
        for (a, b), idx in comparisons:
            expr = f"{op_name}({a}, {b})"
            matches.append((expr, idx) if return_indices else expr)
    return matches

def search_nested_two_operations(inp, return_indices=False):
    """Searches for expressions with two nested binary operators."""
    matches = []
    var_keys = list(inp["variables"].keys())
    const_keys = list(inp["extended_consts"].keys())
    all_keys = var_keys + const_keys

    var_values = list(inp["variables"].values())
    if not var_values: return []

    const_values = [c.expand(len(inp["target"])) for c in inp["extended_consts"].values()]
    value_stack = torch.stack(var_values + const_values)

    index_triplets = torch.cartesian_prod(
        torch.arange(len(var_values)),
        torch.arange(len(all_keys)),
        torch.arange(len(all_keys))
    )
    candidate_triplets = value_stack[index_triplets]
    op_combos = list(itertools.product(inp["binary_ops"].items(), repeat=2))

    for (op1_name, op1), (op2_name, op2) in op_combos:
        # Pattern 1: op1(a, op2(b, c))
        res1 = op1.func(candidate_triplets[:, 0, :], op2.func(candidate_triplets[:, 1, :], candidate_triplets[:, 2, :]))
        comps1 = batch_compare(res1, inp["target"], inp["tol"], inp["min_match"], index_triplets, all_keys)
        for (a, b, c), idx in comps1:
            expr = f"{op1_name}({a}, {op2_name}({b}, {c}))"
            matches.append((expr, idx) if return_indices else expr)

        # Pattern 2: op1(op2(a, b), c)
        res2 = op1.func(op2.func(candidate_triplets[:, 0, :], candidate_triplets[:, 1, :]), candidate_triplets[:, 2, :])
        comps2 = batch_compare(res2, inp["target"], inp["tol"], inp["min_match"], index_triplets, all_keys)
        for (a, b, c), idx in comps2:
            expr = f"{op1_name}({op2_name}({a}, {b}), {c})"
            matches.append((expr, idx) if return_indices else expr)
    return matches

# ==============================================================================
# 5. MAIN DRIVER
# ==============================================================================

def discover_symbolic_rules(df: pd.DataFrame):
    """Main function to drive the discovery of symbolic rules for each parameter."""
    rules = {}
    param_names = list(df.columns)

    for target_col in param_names:
        input_cols = [p for p in param_names if p != target_col]

        search_space = {
            "target": torch.tensor(df[target_col].values, dtype=torch.float32),
            "variables": {inp: torch.tensor(df[inp].values, dtype=torch.float32) for inp in input_cols},
            "simple_consts": {str(c): torch.tensor(c, dtype=torch.float32) for c in [-1., 0., 1.]},
            "extended_consts": {str(c): torch.tensor(c, dtype=torch.float32) for c in LANGUAGE["constants"]},
            "binary_ops": LANGUAGE["binary_ops"],
            "unary_ops": UNARY_OPERATIONS,
            "tol": 0.5,          # Increase to Allow more error per data point
            "min_match": 0.85    # Reduce to Require 85% of data to match, not 95%
        }

        print(f"\n--- Discovering rule for '{target_col}' ---")
        
        search_methods = [
            search_direct_variable,
            search_constants,
            search_unary_operations,
            search_binary_operations,
            search_nested_two_operations,
        ]

        found_rule = None
        for method in search_methods:
            print(f"  -> Running {method.__name__}...")
            matches = method(search_space, return_indices=True)
            if matches:
                expr, _ = matches[0]
                found_rule = expr
                print(f"SUCCESS: {target_col} ≈ {expr}")
                break

        if found_rule:
            rules[target_col] = found_rule
        else:
            print(f"No simple rule discovered for '{target_col}'.")

    return rules

# ==============================================================================
# 6. EXAMPLE USAGE
# ==============================================================================

if __name__ == "__main__":
    # Create a dummy DataFrame for demonstration purposes
    data = {
        'width': [2.0, 3.0, 4.0, 5.0, 6.0],
        'height': [4.0, 6.0, 8.0, 10.0, 12.0],
        'depth': [1.0, 1.0, 1.0, 1.0, 1.0],
        'offset': [-2.0, -3.0, -4.0, -5.0, -6.0]
    }
    analysis_df_dummy = pd.DataFrame(data)

    # Add a more complex rule to the dummy data
    analysis_df_dummy['area_plus_depth'] = (analysis_df_dummy['width'] * analysis_df_dummy['height']) + analysis_df_dummy['depth']

    # Run the discovery process
    rules = discover_symbolic_rules(analysis_df_dummy)

    print("\n==========================")
    print("Final Discovered Rules:")
    for target, rule in rules.items():
        print(f"  {target} = {rule}")
    print("==========================")

In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import pandas as pd

def plot_3d_scatter(df, x_col, y_col, z_col, title="3D Scatter Plot"):
    """
    Creates and displays a 3D scatter plot from a pandas DataFrame.

    Args:
        df (pd.DataFrame): The DataFrame containing the data.
        x_col (str): The name of the column for the X-axis.
        y_col (str): The name of the column for the Y-axis.
        z_col (str): The name of the column for the Z-axis.
        title (str): The title for the plot.
    """
    # Create a new figure and add a 3D subplot
    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection='3d')

    # Create the scatter plot using the specified columns
    ax.scatter(df[x_col], df[y_col], df[z_col], alpha=0.5, marker='o')

    # Set labels and a title for clarity
    ax.set_xlabel(x_col)
    ax.set_ylabel(y_col)
    ax.set_zlabel(z_col)
    plt.title(title)

    # Display the plot
    plt.show()

# --- How to Use It ---

# Assuming 'analysis_df' is your DataFrame with the parameter data
# You can now call the function with the column names you want to plot:
# plot_3d_scatter(
#     df=analysis_df,
#     x_col='param_0',
#     y_col='param_1',
#     z_col='param_2',
#     title="3D Scatter Plot of Parameters"
# )

In [ ]:
pattern_name = "Translate"
raw_records = combined_singletons_detailed.get(pattern_name, [])

analysis_df = prepare_pattern_data(pattern_name, raw_records)
print(analysis_df.head())
plot_3d_scatter(analysis_df, 'param_0', 'param_1', 'param_2', title=f"3D Scatter Plot of Translate Parameters")

# Run the discovery process
print(analysis_df.describe())
rules = discover_symbolic_rules(analysis_df)

print("\n==========================")
print("Final Discovered Rules:")
for target, rule in rules.items():
    print(f"  {target} = {rule}")
print("==========================")

In [ ]:
combined_singletons_detailed.keys()

In [ ]:
analysis_df = prepare_pattern_data("Scale", combined_singletons_detailed.get("Scale_cluster3", []))
print(analysis_df.head())

plot_3d_scatter(analysis_df, 'param_0', 'param_1', 'param_2', title=f"3D Scatter Plot of Scale Parameters")

# Run the discovery process
rules = discover_symbolic_rules(analysis_df)

print("\n==========================")
print("Final Discovered Rules:")
for target, rule in rules.items():
    print(f"  {target} = {rule}")
print("==========================")